In [3]:
import spacy
from itertools import combinations

In [6]:
text1 = """
Microsoft Corporation is an American multinational technology conglomerate headquartered in Redmond,
 Washington. Founded in 1975, the company became influential in the rise of personal computers
   through software like Windows, and has since expanded to Internet services,
     cloud computing, artificial intelligence, video gaming, and other fields.
       Often described as a Big Tech company, Microsoft is the largest software company by revenue,
         one of the most valuable public companies, and one of the most valuable brands globally.
"""
text2 = """
Apple Inc. is an American multinational technology company headquartered in Cupertino, California, in Silicon Valley,
 best known for its consumer electronics, software and online services.
   Founded in 1976 as Apple Computer Company by Steve Jobs, Steve Wozniak and Ronald Wayne,
     the company was incorporated by Jobs and Wozniak as Apple Computer, Inc. the following year. 
It was renamed to its current name in 2007 as the company had expanded its focus from computers to consumer electronics.
"""
text3 = """
Netflix is an American subscription video on-demand over-the-top streaming service. The service primarily 
distributes original and acquired films and television shows from various genres, and it is available
 internationally in multiple languages.

Launched in 2007 nearly a decade after Netflix, Inc. began its pioneering DVD-by-mail movie rental service,
 Netflix is the most-subscribed video on demand streaming media service, with 325 million paid memberships in
   more than 190 countries as of 2026. By 2022, "Netflix Original" productions accounted for half of its
     library in the United States and the namesake company had ventured into other categories,
 such as video game publishing of mobile games through its flagship service. 
"""
text4 = """ 
Visa Inc. is an American multinational payment card services corporation headquartered in San Francisco, California.
 It facilitates electronic funds transfers throughout the world,
 most commonly through Visa-branded credit cards, debit cards and prepaid cards.
Visa does not issue cards, extend credit, or set rates and fees for consumers; rather,
 Visa provides financial institutions with Visa-branded payment products that they then use to 
offer credit, debit, prepaid and cash access programs to their customers. In 2025, Visa's global network
 (known as VisaNet) processed 257.5 billion transactions worth US$14.2 trillion.

Visa was founded in 1958 by Bank of America (BofA) as the BankAmericard credit card program.
In response to competitor Master Charge (now Mastercard), BofA began to license the BankAmericard
 program to other financial institutions in 1966.[8] By 1970, BofA gave up direct control of
   the BankAmericard program, forming a cooperative with the other various BankAmericard issuer banks
     to take over its management. It was then renamed Visa in 1976.
"""

text5 = """
Google LLC is an American multinational technology corporation focused on information technology,
 online advertising, search engine technology, email, cloud computing, software, quantum computing,
   e-commerce, consumer electronics, and artificial intelligence (AI).[9] It has been referred to as
     "the most powerful company in the world" by BBC,[10] and is one of the world's most valuable brands.
       Google's parent company Alphabet Inc. has been described as a Big Tech company.
Google was founded on September 4, 1998, by American computer scientists Larry Page and Sergey Brin. 
Together, they own about 14% of its publicly listed shares and control 56% of its stockholder voting
 power through super-voting stock. 
"""

text6 = """
The Federal Reserve System (often shortened to the Federal Reserve, or simply the Fed) is the central banking system of the
 United States. It was created on December 23, 1913, with the enactment of the Federal Reserve Act,
   after a series of financial panics (particularly the panic of 1907) led to the desire for central
     control of the monetary system in order to alleviate financial crises.[list 1] Although an instrument
       of the U.S. government, the Federal Reserve System considers itself "an independent central bank because
         its monetary policy decisions do not have to be approved by the president or by anyone else in the
           executive or legislative branches of government, it does not receive funding appropriated by Congress, and the terms of the members of the board of governors span multiple presidential and congressional terms."
Over the years, events such as the Great Depression in the 1930s and the Great Recession during the 2000s have led to the expansion of the roles and responsibilities of the Federal Reserve System.
"""

In [7]:
nlp = spacy.load("en_core_web_sm")
texts = [text1, text2, text3, text4, text5, text6]

In [ ]:
RELATION_TYPES = ["FOUNDED_BY", "FOUNDED_IN_YEAR", "LOCATED_IN", "NO_RELATION"]

relation_keywords = {
    "FOUNDED_BY": ["founded", "created", "established", "led by"],
    "FOUNDED_IN_YEAR": ["in", "on"],
    "LOCATED_IN": ["in", "at"]
}

def extract_entities(text):
    doc = nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    return doc, entities

def get_entity_pairs(entities):
    pairs = list(combinations(range(len(entities)), 2))
    return [(entities[i], entities[j]) for i, j in pairs]

def classify_relation(ent1, ent2, text):
    label1, label2 = ent1[1], ent2[1]
    text_lower = text.lower()
    
    if label1 == "ORG" and label2 == "PERSON":
        if any(kw in text_lower for kw in relation_keywords["FOUNDED_BY"]):
            return "FOUNDED_BY"
    if label1 == "ORG" and label2 == "DATE":
        if any(kw in text_lower for kw in relation_keywords["FOUNDED_IN_YEAR"]):
            return "FOUNDED_IN_YEAR"
    if label1 == "ORG" and label2 in ["GPE", "LOCATION"]:
        if any(kw in text_lower for kw in relation_keywords["LOCATED_IN"]):
            return "LOCATED_IN"
    
    return "NO_RELATION"

In [9]:
results = []
for idx, text in enumerate(texts):
    print(f"\n{'='*60}")
    print(f"TEXT {idx + 1}: {text}")
    print(f"{'='*60}")
    
    doc, entities = extract_entities(text)
    print(f"\nEXTRACTED ENTITIES:")
    for ent_text, ent_type in entities:
        print(f"  - {ent_text:30s} [{ent_type}]")
    
    pairs = get_entity_pairs(entities)
    print(f"\nENTITY PAIRS ({len(pairs)} total):")
    
    relations = []
    for ent1, ent2 in pairs:
        relation = classify_relation(ent1, ent2, text)
        relations.append({
            'Entity1': ent1[0],
            'Type1': ent1[1],
            'Entity2': ent2[0],
            'Type2': ent2[1],
            'Relation': relation
        })
        status = "✓" if relation != "NO_RELATION" else "✗"
        print(f"  {status} {ent1[0]:20s} --[{relation:15s}]-- {ent2[0]}")
    
    results.append({
        'text_id': idx + 1,
        'text': text,
        'entities': entities,
        'relations': relations
    })


TEXT 1: 
Microsoft Corporation is an American multinational technology conglomerate headquartered in Redmond,
 Washington. Founded in 1975, the company became influential in the rise of personal computers
   through software like Windows, and has since expanded to Internet services,
     cloud computing, artificial intelligence, video gaming, and other fields.
       Often described as a Big Tech company, Microsoft is the largest software company by revenue,
         one of the most valuable public companies, and one of the most valuable brands globally.


EXTRACTED ENTITIES:
  - Microsoft Corporation          [ORG]
  - American                       [NORP]
  - Redmond                        [GPE]
  - Washington                     [GPE]
  - 1975                           [DATE]
  - Windows                        [LOC]
  - Big Tech                       [ORG]
  - Microsoft                      [ORG]
  - one                            [CARDINAL]
  - one                            [CARD

In [12]:
print(f"\n\n{'='*60}")
print("DETAILED ANALYSIS - TEXT 1")
print(f"{'='*60}")
first_result = results[0]
print(f"Text: {first_result['text']}\n")

for rel in first_result['relations']:
    if rel['Relation'] != "NO_RELATION":
        print(f"✓ VALID: {rel['Entity1']} --[{rel['Relation']}]-- {rel['Entity2']}")
    else:
        print(f"✗ NO RELATION: {rel['Entity1']} -- {rel['Entity2']}")

print("\nERROR ANALYSIS:")
print("- Many NO_RELATION pairs occur because entities are semantically unrelated")
print("- Simple keyword matching may fail with complex sentence structures")
print("- Context dependency requires more sophisticated NLP models for accurate classification")



DETAILED ANALYSIS - TEXT 1
Text: 
Microsoft Corporation is an American multinational technology conglomerate headquartered in Redmond,
 Washington. Founded in 1975, the company became influential in the rise of personal computers
   through software like Windows, and has since expanded to Internet services,
     cloud computing, artificial intelligence, video gaming, and other fields.
       Often described as a Big Tech company, Microsoft is the largest software company by revenue,
         one of the most valuable public companies, and one of the most valuable brands globally.


✗ NO RELATION: Microsoft Corporation -- American
✓ VALID: Microsoft Corporation --[LOCATED_IN]-- Redmond
✓ VALID: Microsoft Corporation --[LOCATED_IN]-- Washington
✓ VALID: Microsoft Corporation --[FOUNDED_IN_YEAR]-- 1975
✗ NO RELATION: Microsoft Corporation -- Windows
✗ NO RELATION: Microsoft Corporation -- Big Tech
✗ NO RELATION: Microsoft Corporation -- Microsoft
✗ NO RELATION: Microsoft Corporation -- o

W analizowanym tekście model poprawnie wykrył kluczowe encje, takie jak Microsoft Corporation (ORG), Redmond i Washington (LOCATION) oraz 1975 (DATE) Na ich podstawie poprawnie zidentyfikowano relacje LOCATED_IN (Microsoft Corporation – Redmond / Washington) oraz FOUNDED_IN_YEAR (Microsoft Corporation – 1975), które są zgodne z treścią tekstu.

Większość wygenerowanych par encji została poprawnie oznaczona jako NO_RELATION. Wynika to z faktu, że sama współobecność encji w tym samym zdaniu nie implikuje istnienia relacji semantycznej, np. pary takie jak Microsoft Corporation – Windows czy American – Washington nie opisują jednoznacznej relacji zdefiniowanej w przyjętym zbiorze typów relacji.

Pojawiające się błędne lub wątpliwe przypadki wynikają głównie z:

* **nadmiarowej ekstrakcji encji** (np. tokeny takie jak „one”),
* **ograniczeń prostych reguł opartych na słowach kluczowych**, które nie uwzględniają pełnej struktury składniowej zdania,
* **braku głębszego rozumienia kontekstu**, np. relacji pośrednich lub opisowych.

Podsumowując, system poprawnie identyfikuje podstawowe, jawnie wyrażone relacje, jednak generuje wiele kandydatów bez sensownej relacji, co pokazuje, że etap klasyfikacji relacji jest znacznie trudniejszy niż samo rozpoznawanie encji i wymaga bardziej zaawansowanych modeli lub dodatkowych heurystyk.